In [1]:
# Self Attention 설명시 제공되는 이미지를 실제로 코드로 작업하기
import numpy as np
import math

# 1) Self-Attention 예제 문장
tokens = [
    "The", "animal", "didn't", "cross", "the", "street",
    "because", "it", "was", "too", "tired", "."
]

# 2) 임베딩 차원 및 가중치 행렬 설정
d_model = 4  # 임베딩 차원, 즉 각 단어가 표현되는 벡터의 길이를 의미. 동작 원리 설명용
np.random.seed(1)
embeddings = np.random.rand(len(tokens), d_model) # (12, 4) 모양의 난수 행렬을 생성
    # → 각 단어마다 랜덤한 4차원 벡터를 생성하여 임베딩으로 사용
    # 무슨 역할? 이 줄은 각 단어에 해당하는 임베딩 벡터를 임의로 생성.
    # 단어	임베딩 벡터 예시 (4차원)  :  "The"	[0.12, 0.55, 0.87, 0.02]
    # Q, K, V 벡터를 계산하는 데 사용

# Query, Key, Value를 만드는 가중치 행렬 (실제론 학습됨). 4×4 크기의 난수 행렬을 생성
Wq = np.random.rand(d_model, d_model)  # Wq는 Query 벡터를 만들기 위한 가중치 행렬
Wk = np.random.rand(d_model, d_model)
Wv = np.random.rand(d_model, d_model)

# 3) Q, K, V 계산
# 각 단어 임베딩을 Query 벡터로 선형 변환(linear transformation) 하는 것
Q = embeddings.dot(Wq)  # Self-Attention 계산에서 Query 벡터 Q를 만드는 단계
K = embeddings.dot(Wk)
V = embeddings.dot(Wv)
  # Q: Query 벡터 (각 입력 토큰이 어떤 정보에 주목하고 싶은지)
  # K: Key 벡터 (각 입력 토큰이 어떤 정보를 가지고 있는지)
  # V: Value 벡터 (각 토큰이 실제로 가진 정보)

# 4) Scaled Dot-Product Attention 함수
def scaled_dotAttentionFunc(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q.dot(K.T) / math.sqrt(d_k)  # Query와 Key의 유사도를 구한다.

    #  Softmax → Attention Weight로 변환
    # np.max() : numerical stability를 위한 보정
    exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))

    # weights는 각 단어가 다른 단어에 주는 가중치 (0~1의 값, 합=1)
    weights = exp_scores / exp_scores.sum(axis=1, keepdims=True)

    # 가중합 → Context 벡터 생성
    # Attention weight와 Value 벡터를 가중합(각 단어의 문맥 벡터 (정보를 종합한 최종 결과))
    context = weights.dot(V)

    # weights: 어떤 단어가 어떤 단어에 집중하는지 나타냄
    # context: 각 단어에 대해 Self-Attention으로 추출된 문맥 정보
    return weights, context

# 5) Attention 수행 : Self-Attention에서 실제 계산을 수행
weights, context = scaled_dotAttentionFunc(Q, K, V)

# 6) 'it' 토큰에 대한 Attention 결과 출력
  # Self-Attention 결과에서 "it" 이라는 단어가 어떤 단어들에 주의를 기울였는지,
  # 그리고 그로부터 어떤 문맥(Context) 벡터를 얻었는지를 추출

# "it"이 문장 속 몇 번째에 있는지를 찾는다.
# 예: tokens = ["The", "animal", ..., "it", "was", "too", ...] 라면 "it"은 인덱스 7일 수 있다.
i_it = tokens.index("it")

# Self-Attention 결과인 weights 중 "it" 위치에 해당하는 attention 가중치 행을 추출.
  # 즉, "it"이 문장 내 다른 단어 각각에 얼마나 집중하는지를 나타낸다.
  # it_weights는 길이가 len(tokens)인 벡터 (예: 12차원 softmax 값)
it_weights = weights[i_it]
# 쉽게 예를 들면
#     단어	           The	   animal	...	 tired
# attention 가중치	   0.05	   0.70 	...	  0.10
#   → "it"은 "animal"에게 가장 집중하고 있음 → 이걸 바탕으로 문맥 벡터를 계산
#   → it_context = weighted sum of V

print("Tokens:", tokens)

print("\nAttention weights for 'it':")
for tok, w in zip(tokens, it_weights):
    print(f"  {tok:>8}: {w:.3f}")  # The: 0.098   animal: 0.082  ...

# "it"의 최종 문맥 벡터(Context Vector)이다. 이 벡터는 V(value)들을 it_weights로 가중합한 값임.
# 크기는 d_model (예: 4차원, [0.83, 1.02, 0.61, 0.44] 같은 형태)
it_context = context[i_it]
print("\nContext vector for 'it':", np.round(it_context, 3))  # 'it'이 주변 단어로부터 얻은 의미 정보가 압축된 결과

# 7) Query/Key/Value 예시: 단어 'student'
word = "student"
# 랜덤 임베딩 할당 (예시)
emb_student = np.random.rand(d_model)
Q_st = emb_student.dot(Wq)
K_st = emb_student.dot(Wk)
V_st = emb_student.dot(Wv)

# "student"라는 단어의 임베딩을 기반으로 만든 Self-Attention 입력 벡터들
print("\nQ_student:", np.round(Q_st, 3))
print("K_student:", np.round(K_st, 3))
print("V_student:", np.round(V_st, 3))

Tokens: ['The', 'animal', "didn't", 'cross', 'the', 'street', 'because', 'it', 'was', 'too', 'tired', '.']

Attention weights for 'it':
       The: 0.063
    animal: 0.040
    didn't: 0.084
     cross: 0.082
       the: 0.056
    street: 0.132
   because: 0.077
        it: 0.070
       was: 0.096
       too: 0.106
     tired: 0.132
         .: 0.064

Context vector for 'it': [1.127 1.48  0.993 1.281]

Q_student: [0.446 0.526 0.73  0.901]
K_student: [1.16  1.038 1.582 0.993]
V_student: [1.232 1.436 1.176 1.018]
